In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### **Read & Load the Documents**

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
loader_harrypotter  = PyPDFLoader("../harrypotter_1.pdf")
documnet_harrypotter = loader_harrypotter.load()

In [4]:
print(len(documnet_harrypotter))

250


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100,
    length_function = len,
    is_separator_regex = False,
)

In [6]:
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)
# texts = [doc.page_content for doc in text_harrypotter]
len(text_harrypotter)

1155

#### **Load the Embeddings Model**

In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### **Vector DB Setup**

In [8]:
from langchain_community.vectorstores import FAISS

In [9]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

In [10]:
print(text_harrypotter[2].metadata)

{'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 1, 'page_label': '2'}


In [11]:
for id, text in enumerate(text_harrypotter):
    text.metadata["id"] = id

In [12]:
print(text_harrypotter[2].metadata)

{'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 1, 'page_label': '2', 'id': 2}


In [13]:
text_harrypotter[:5]

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 1, 'page_label': '2', 'id': 0}, page_content="1\nHarry Potter and the Sorcerer's Stone\nCHAPTER ONE\nTHE BOY WHO LIVED\nMr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the last\npeople you'd expect to be involved in anything strange or mysterious,\nbecause they just didn't hold with such nonsense.\nMr. Dursley was the director of a firm called Grunnings, which made\ndrills. He was a big, beefy man with hardly any neck, although he did"),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdat

In [14]:
retriever = FAISS.from_documents(text_harrypotter, embeddings).as_retriever(
    search_kwargs={"k": 10}
)

In [15]:
query = "Who gave Harry his first broomstick?"

docs = retriever.invoke(query)

In [16]:
pretty_print_docs(docs)

Document 1:

"That's a broomstick," he said, throwing it back to Harry with a mixture
of jealousy and spite on his face. "You'll be in for it this time,
Potter, first years aren't allowed them."
Ron couldn't resist it.
"It's not any old broomstick," he said, "it's a Nimbus Two Thousand.
What did you say you've got at home, Malfoy, a Comet Two Sixty?" Ron
grinned at Harry. "Comets look flashy, but they're not in the same
league as the Nimbus."
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 131, 'page_label': '132', 'id': 614}
----------------------------------------------------------------------------------------------------
Document 2:

learning to play that night. He bolted his dinner that

#### **Load LLM**

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0
)

#### **Setup Re-ranker**

In [18]:
# %pip install flashrank

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from flashrank import Ranker
from flashrank import RerankRequest  # noqa: F401

In [22]:
ranker = Ranker(model_name="ms-marco-MultiBERT-L-12")
compressor = FlashrankRerank(client=ranker, top_n=4)

In [23]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [24]:
query = "Who gave Harry his first broomstick?"
compressed_docs = compression_retriever.invoke(query)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [25]:
len(compressed_docs)

4

In [26]:
compressed_docs

[Document(metadata={'id': 538, 'relevance_score': np.float32(0.99897534), 'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 114, 'page_label': '115'}, page_content="players move.\nNeville had never been on a broomstick in his life, because his\ngrandmother had never let him near one. Privately, Harry felt she'd had\ngood reason, because Neville managed to have an extraordinary number of\naccidents even with both feet on the ground.\nHermione Granger was almost as nervous about flying as Neville was. This\nwas something you couldn't learn by heart out of a book -- not that she\nhadn't tried. At breakfast on Thursday she bored them all stupid with"),
 Document(metadata={'id': 555, 'relevance_score': np.flo

In [27]:
print([doc.metadata["id"] for doc in compressed_docs])

[538, 555, 613, 619]


In [28]:
pretty_print_docs(compressed_docs)

Document 1:

players move.
Neville had never been on a broomstick in his life, because his
grandmother had never let him near one. Privately, Harry felt she'd had
good reason, because Neville managed to have an extraordinary number of
accidents even with both feet on the ground.
Hermione Granger was almost as nervous about flying as Neville was. This
was something you couldn't learn by heart out of a book -- not that she
hadn't tried. At breakfast on Thursday she bored them all stupid with
Metadata: {'id': 538, 'relevance_score': np.float32(0.99897534), 'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2001-02-13T16:47:14+00:00', 'subject': 'Harry Potter', 'author': 'J.K. Rowling', 'moddate': '2005-11-26T18:01:39+02:00', 'title': "Harry Potter, Book 1; The Sorcerer's Stone", 'source': '../harrypotter_1.pdf', 'total_pages': 250, 'page': 114, 'page_label': '115'}
-----------------------------------------------------------------------------

In [29]:
from langchain.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [31]:
query = "Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?"

response = chain.invoke(query)
response

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'query': 'Why does Uncle Vernon go to extreme lengths to prevent Harry from reading his letters?',
 'result': "Uncle Vernon goes to extreme lengths to prevent Harry from reading his letters because he is deeply concerned about the content of the letters and what they might mean for Harry. The letters are from Hogwarts, indicating that Harry is a wizard, which Uncle Vernon vehemently opposes. He wants to keep Harry from discovering his true identity and the magical world, fearing it will disrupt their ordinary, non-magical life. Uncle Vernon's reaction to the letters shows his anxiety and determination to control the situation, even resorting to extreme measures to keep Harry from accessing them."}